In [1]:
import pandas as pd
import numpy as np  
import os

In [2]:
# Load Tira dataset
df_tira = pd.read_csv(
    "Input/tira_campaign_data_with_nulls.csv"
)

# Basic inspection
print("Tira dataset shape:", df_tira.shape)

print("\nColumns:")
print(df_tira.columns.tolist())

print("\nData types:")
display(df_tira.dtypes)

Tira dataset shape: (55555, 16)

Columns:
['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue', 'Acquisition_Cost', 'ROI', 'Language', 'Engagement_Score', 'Customer_Segment', 'Date']

Data types:


Campaign_ID             str
Campaign_Type           str
Target_Audience         str
Duration            float64
Channel_Used            str
Impressions         float64
Clicks              float64
Leads               float64
Conversions         float64
Revenue             float64
Acquisition_Cost    float64
ROI                 float64
Language                str
Engagement_Score    float64
Customer_Segment        str
Date                    str
dtype: object

In [3]:
# Check missing values in Tira dataset

missing_values = df_tira.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (missing_values / len(df_tira) * 100).round(2)
})

display(
    missing_summary[
        missing_summary["Missing_Count"] > 0
    ]
)

,Missing_Count,Missing_Percentage
Campaign_ID,2733,4.92
Campaign_Type,2709,4.88
Target_Audience,2664,4.80
Duration,2665,4.80
Channel_Used,2733,4.92
Impressions,2723,4.90
Clicks,2730,4.91
Leads,2657,4.78
Conversions,2660,4.79
Revenue,2729,4.91


In [4]:
# Check duplicate records

print("Duplicate rows:", df_tira.duplicated().sum())
print("Duplicate Campaign_IDs:", df_tira["Campaign_ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Campaign_IDs: 2732


In [5]:
# Handle missing Revenue and Acquisition_Cost

df_tira["Revenue"] = df_tira["Revenue"].fillna(0)

acquisition_cost_mean = df_tira["Acquisition_Cost"].mean()

df_tira["Acquisition_Cost"] = (
    df_tira["Acquisition_Cost"]
    .fillna(acquisition_cost_mean)
)

print("Revenue missing values:",
      df_tira["Revenue"].isna().sum())

print("Acquisition_Cost missing values:",
      df_tira["Acquisition_Cost"].isna().sum())

print("Acquisition_Cost mean used:",
      acquisition_cost_mean)

Revenue missing values: 0
Acquisition_Cost missing values: 0
Acquisition_Cost mean used: 374.9208677818732


In [6]:
# Fill remaining numerical columns with mean
# ROI is intentionally excluded

other_numerical_columns = [
    "Duration",
    "Impressions",
    "Clicks",
    "Leads",
    "Conversions",
    "Engagement_Score"
]

for col in other_numerical_columns:
    df_tira[col] = df_tira[col].fillna(
        df_tira[col].mean()
    )

print("Remaining numerical columns handled:")
print(other_numerical_columns)

print("\nMissing values:")
display(
    df_tira[other_numerical_columns].isna().sum()
)

Remaining numerical columns handled:
['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Engagement_Score']

Missing values:


Duration            0
Impressions         0
Clicks              0
Leads               0
Conversions         0
Engagement_Score    0
dtype: int64

In [7]:
# Fill categorical columns with mode

categorical_columns = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Language",
    "Customer_Segment"
]

for col in categorical_columns:
    mode_value = df_tira[col].mode()[0]
    df_tira[col] = df_tira[col].fillna(mode_value)

print("Categorical columns handled using mode:")
print(categorical_columns)

print("\nMissing values:")
display(
    df_tira[categorical_columns].isna().sum()
)

Categorical columns handled using mode:
['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment']

Missing values:


Campaign_Type       0
Target_Audience     0
Channel_Used        0
Language            0
Customer_Segment    0
dtype: int64

In [8]:
# Convert Date column to datetime

df_tira["Date"] = pd.to_datetime(
    df_tira["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Date datatype:", df_tira["Date"].dtype)
print(
    "Missing dates after conversion:",
    df_tira["Date"].isna().sum()
)

display(df_tira[["Date"]].head())

Date datatype: datetime64[us]
Missing dates after conversion: 2690


,Date
0,2024-10-30
1,2024-09-11
2,2024-08-28
3,2024-07-27
4,2024-12-19


In [9]:
# Add Company Name

df_tira["Company_Name"] = "Tira"

# Calculate Profit

df_tira["Profit"] = (
    df_tira["Revenue"] -
    df_tira["Acquisition_Cost"]
)

# Calculate Profit Flag

df_tira["Profit_Flag"] = np.where(
    df_tira["Profit"] > 0,
    "Profit",
    "Loss"
)

# Calculate ROI

df_tira["Calculated_ROI"] = (
    df_tira["Profit"] /
    df_tira["Acquisition_Cost"]
)

print("Feature engineering completed successfully.")

display(
    df_tira[
        [
            "Company_Name",
            "Revenue",
            "Acquisition_Cost",
            "Profit",
            "Profit_Flag",
            "Calculated_ROI"
        ]
    ].head(10)
)

Feature engineering completed successfully.


,Company_Name,Revenue,Acquisition_Cost,Profit,Profit_Flag,Calculated_ROI
0,Tira,222768.0,129.720000,2.226383e+05,Profit,1716.298797
1,Tira,199168.0,316.360000,1.988516e+05,Profit,628.561259
2,Tira,433620.0,374.920868,4.332451e+05,Profit,1155.564057
3,Tira,571872.0,161.130000,5.717109e+05,Profit,3548.134239
4,Tira,1477936.0,82.070000,1.477854e+06,Profit,18007.236871
5,Tira,185668.0,194.210000,1.854738e+05,Profit,955.016683
6,Tira,2691144.0,32.980000,2.691111e+06,Profit,81598.272286
7,Tira,1230558.0,374.920868,1.230183e+06,Profit,3281.180603
8,Tira,804784.0,43.890000,8.047401e+05,Profit,18335.386421
9,Tira,890704.0,190.580000,8.905134e+05,Profit,4672.648861


In [10]:
# Check remaining missing values

missing_after_cleaning = df_tira.isnull().sum()

display(
    missing_after_cleaning[
        missing_after_cleaning > 0
    ]
)

Campaign_ID    2733
ROI            2742
Date           2690
dtype: int64

In [11]:
# Multi-label encoding for Channel_Used

channels = [
    "Google",
    "WhatsApp",
    "YouTube",
    "Email",
    "Instagram",
    "Facebook"
]

for channel in channels:
    df_tira[f"Channel_{channel}"] = (
        df_tira["Channel_Used"]
        .str.split(",")
        .apply(
            lambda x: int(
                channel in [item.strip() for item in x]
            )
        )
    )

print("Multi-label encoding completed successfully.")

display(
    df_tira[
        [
            "Channel_Used",
            "Channel_Google",
            "Channel_WhatsApp",
            "Channel_YouTube",
            "Channel_Email",
            "Channel_Instagram",
            "Channel_Facebook"
        ]
    ].head(10)
)

Multi-label encoding completed successfully.


,Channel_Used,Channel_Google,Channel_WhatsApp,Channel_YouTube,Channel_Email,Channel_Instagram,Channel_Facebook
0,Email,0,0,0,1,0,0
1,YouTube,0,0,1,0,0,0
2,Email,0,0,0,1,0,0
3,Google,1,0,0,0,0,0
4,"Facebook, Email, WhatsApp",0,1,0,1,0,1
5,"Google, Facebook",1,0,0,0,0,1
6,"Email, YouTube, WhatsApp",0,1,1,1,0,0
7,"Google, Facebook, Instagram",1,0,0,0,1,1
8,YouTube,0,0,1,0,0,0
9,"Email, WhatsApp, Facebook",0,1,0,1,0,1


In [12]:
# Validate channel indicator totals

channel_columns = [
    "Channel_Google",
    "Channel_WhatsApp",
    "Channel_YouTube",
    "Channel_Email",
    "Channel_Instagram",
    "Channel_Facebook"
]

print("Number of campaigns using each channel:")

display(
    df_tira[channel_columns]
    .sum()
    .sort_values(ascending=False)
)

Number of campaigns using each channel:


Channel_Email        20341
Channel_Instagram    17786
Channel_Facebook     17677
Channel_Google       17650
Channel_WhatsApp     17624
Channel_YouTube      17605
dtype: int64

In [13]:
# Validate that every campaign has at least one encoded channel

channel_indicator_total = (
    df_tira[channel_columns].sum(axis=1)
)

print(
    "Campaigns with no encoded channel:",
    (channel_indicator_total == 0).sum()
)

print(
    "Campaigns with one or more encoded channels:",
    (channel_indicator_total >= 1).sum()
)

print(
    "Total campaigns:",
    len(df_tira)
)

Campaigns with no encoded channel: 0
Campaigns with one or more encoded channels: 55555
Total campaigns: 55555


In [14]:
# Save cleaned and feature-engineered Tira dataset

output_path = (
    r"D:\Data Science\vscode"
    r"\Marketing_Campaign_Performance_Prediction"
    r"\Output\Tira_Feature_Engineered.csv"
)

df_tira.to_csv(output_path, index=False)

print("Tira dataset saved successfully.")
print("File:", output_path)
print("Shape:", df_tira.shape)

Tira dataset saved successfully.
File: D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\Tira_Feature_Engineered.csv
Shape: (55555, 26)
